# 01 - Business Understanding
## Vehicle Safety Intelligence: linking police-reported crashes and owner complaints

**CRISP-DM phase:** Business Understanding
**Data:** NHTSA Crash Report Sampling System (CRSS) 2020-2024 (structured, 28 relational tables per year) and NHTSA Vehicle Owner Complaints 1995-2026 (structured metadata + unstructured narratives).

## 1. Problem definition
Vehicle-safety analysts, such as those in NHTSA's Office of Defects Investigation, must decide which vehicles and components deserve closer scrutiny, and which individual reports describe serious harm. They face two very different evidence streams:

* **Owner complaints** arrive as free text, more than 60,000 per year. Each must be routed to the affected component and screened for reports of crashes, fires, injuries or deaths. Raw complaint counts are misleading because popular vehicles attract more complaints simply because more of them are on the road.
* **Police-reported crashes** describe what happened on the road in hundreds of coded fields. They show which circumstances lead to injury and serious harm, and how often each vehicle is involved in crashes.

Neither source answers the screening question alone. This project builds **one decision-support system** that uses both: text models triage complaints, crash models quantify severity, and the two are linked at **make + model + model year** to put complaint volumes on a fair, exposure-adjusted footing.

## 2. Research questions
| # | Question | Data | Answered by |
|---|---|---|---|
| RQ1 | Which crash, road, vehicle and person characteristics distinguish **injury crashes** from property-damage-only crashes, and how well can they be predicted? | CRSS (structured) | Model **S1** + EDA statistical tests |
| RQ2 | Which characteristics are associated with **serious or fatal** injury, and can those crashes be identified despite class imbalance? | CRSS (structured) | Model **S2** + EDA |
| RQ3 | Can complaint narratives be **routed automatically** to the affected component group(s)? | Complaints (unstructured) | Model **U1** (multi-label) |
| RQ4 | Can narratives that describe a **crash, fire, injury or death** be flagged for priority review? | Complaints (unstructured) | Model **U2** |
| RQ5 | When complaints are adjusted for on-road exposure estimated from CRSS, which vehicles stand out, and does this differ from raw counts? | Both, linked | Integrated vehicle profiles |

Every documented field is considered. Fields are excluded only when they identify a record, encode the outcome (leakage) or describe the survey design, and each exclusion is recorded with its reason (`outputs/tables/crss_excluded_fields.csv`).

## 3. Stakeholders and decisions
| Stakeholder | Decision supported | Explicit restriction |
|---|---|---|
| Complaint triage analyst | Route each narrative to component specialists; prioritise reports of harm | A recommendation is not proof of a defect |
| Defect screening analyst | Decide which make/model/year/component combinations merit review | Exposure-adjusted rates are screening signals, not defect rates |
| Road-safety analyst | Understand which crash circumstances drive injury and serious injury | Associations do not establish causation or liability |
| Data-science reviewer | Assess reproducibility, leakage control and temporal generalisation | Test periods cannot influence model selection |

## 4. Units of analysis and targets
**S1 / S2 (CRSS):** one sampled crash (`CASENUM` within year). All 28 tables, including the VIN-decode vehicle specifications, are aggregated to crash level without duplicating crashes.
* S1 target: injury crash (KABCO C, B, A, K or injured-severity-unknown) vs property-damage-only (O).
* S2 target: serious or fatal (A or K) vs other known outcomes (O, C, B).
* `MAX_SEV`, injury counts, person/vehicle injury severities, hospital transport and the sampling stratum (defined partly by severity) are excluded as leakage.

**U1 / U2 (complaints):** one complaint (`ODINO`); repeated component rows are combined.
* U1 target: the set of harmonised component groups (19 groups mapped from `COMPDESC`; 22% of complaints name more than one, so the task is multi-label).
* U2 target: the complaint's structured crash / fire / injury / death flags. The narrative is the only predictor.

**Integration:** normalised make + model + model year. No record-level join is made or implied.

## 5. Evaluation design fixed before modelling
| Dataset | Train | Validation (tuning, thresholds) | Test (scored once) |
|---|---|---|---|
| CRSS | 2020-2022 | 2023 | 2024 |
| Complaints | received 2010-2021 | 2022-2023 | 2024-2026 |

Encoders, vocabularies, TF-IDF and code lists are fitted on training data only. Exact-duplicate narratives are removed so no validation or test narrative repeats a training one. Overfitting is checked with train-validation gaps and learning curves; drift with validation-test gaps.

## 6. Model families compared
* **Structured (S1, S2):** dummy baseline, Logistic Regression, Decision Tree, Random Forest, Histogram Gradient Boosting; with and without post-crash fields; with and without class weighting (S2).
* **Unstructured (U1, U2):** dummy baseline, Complement Naive Bayes, Logistic Regression, Linear SVM on training-fitted TF-IDF word uni- and bi-grams.

Selection uses validation results only. Metrics: ROC-AUC and PR-AUC, precision, recall and F1 at validated thresholds, calibration (S1/S2), macro/micro F1 and per-component results (U1).

## 7. Success criteria
* Every model beats its dummy baseline on untouched test data.
* Train-validation gaps and learning curves show no material overfitting; validation-test gaps quantify drift.
* At least two structured and two unstructured models are trained and compared; the selected models are deployed in a web application with explanations, warnings and a human-review disclaimer.
* The integrated analysis demonstrates a result that neither dataset could produce alone (exposure-adjusted complaint rates).

## 8. Risks and controls
| Risk | Control |
|---|---|
| CRSS annual code changes | Harmonise only after checking the analytical and annual coding manuals |
| Sampling design ignored | Distinguish weighted population estimates from unweighted modelling counts |
| Crash target leakage | Maintain an explicit excluded-field register and verify pipeline inputs |
| Repeated complaint rows | Partition and evaluate by `ODINO` |
| Duplicate complaint narratives | Audit exact/near duplicates before splitting |
| Voluntary-report bias | Never interpret complaint counts as defect rates |
| Temporal drift | Use chronological validation/test periods and monitoring |
| Overconfident automation | Display limitations and retain human decision authority |

## 9. CRISP-DM hand-off
Notebook 02 verifies that the data can support these questions (inventory, joins, label availability, linkage coverage, statistical tests). Notebook 03 builds the analytical tables, 04-05 train and evaluate the models, and 06 documents the deployed application.